# Tiny Chat 20M - Colab GPU Training

This notebook runs the existing PyTorch training scripts on a Colab GPU. Checkpoints are stored in Google Drive when Drive is mounted.

In [ ]:
import os
import sys
import torch

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Select Runtime > Change runtime type > T4 GPU in Colab.')

## Mount Drive and install the project

The project is expected at `/content/drive/MyDrive/ai`. Change `DRIVE_PROJECT_DIR` if your folder is elsewhere.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = ''
PROJECT_DIR = '/content/drive/MyDrive/ai'

from google.colab import drive
print('Mounting Google Drive...')
drive.mount('/content/drive', force_remount=True)

project_path = Path(PROJECT_DIR)
if not project_path.exists():
    raise FileNotFoundError(
        f"Could not find '{PROJECT_DIR}'. Make sure the folder is named exactly 'ai' and is in My Drive."
    )
if not (project_path / 'pyproject.toml').exists():
    raise FileNotFoundError(
        f"Found the folder, but pyproject.toml is missing: {PROJECT_DIR}"
    )

os.environ['PROJECT_DIR'] = PROJECT_DIR
os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
print('Successfully installed project from:', PROJECT_DIR)

In [ ]:
USE_DRIVE = True
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, 'checkpoints')
if USE_DRIVE:
    CHECKPOINT_DIR = '/content/drive/MyDrive/tiny_chat_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
CHECKPOINT = os.path.join(CHECKPOINT_DIR, 'chat_model.pt')
print('Checkpoint:', CHECKPOINT)

## Prepare Hugging Face data

This downloads and formats the UltraChat dataset. It may take time the first time only.

In [ ]:
%cd $PROJECT_DIR
!python -m tiny_chat.data

## Configure training

Use a smaller `MAX_CHARS` for a quick experiment. Use 20000000 or more for a stronger run if the runtime has enough memory.

In [ ]:
STEPS = 5000
BATCH_SIZE = 16
MAX_CHARS = 20000000
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print({'steps': STEPS, 'batch_size': BATCH_SIZE, 'max_chars': MAX_CHARS, 'device': DEVICE})

## GPU smoke test

Run this before committing to a long training job.

In [ ]:
%cd $PROJECT_DIR
!python -m tiny_chat.train --steps 2 --batch-size 2 --max-chars 100000 --device $DEVICE --checkpoint $CHECKPOINT

In [ ]:
%cd $PROJECT_DIR
!python -m tiny_chat.train --steps $STEPS --batch-size $BATCH_SIZE --max-chars $MAX_CHARS --device $DEVICE --checkpoint $CHECKPOINT

## Chat with the checkpoint

Run this cell, then type messages in the terminal input. Use `/quit` to exit.

In [ ]:
%cd $PROJECT_DIR
!python -m tiny_chat.chat --checkpoint $CHECKPOINT --device $DEVICE --temperature 0.5 --top-k 40 --top-p 0.9 --repetition-penalty 1.1 --tokens 80